In [ ]:
# Bài 19 — Sentiment Classification trên IMDb bằng pretrained BERT
%pip -q install datasets transformers scikit-learn

import random, time, numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

imdb = load_dataset("stanfordnlp/imdb")

def balanced_sample(ds, n, seed):
    n_each = n // 2
    labels = np.asarray(ds["label"])
    rng = np.random.default_rng(seed)
    chosen = []
    for label in (0, 1):
        idx = np.flatnonzero(labels == label)
        rng.shuffle(idx)
        chosen.extend(idx[:n_each].tolist())
    rng.shuffle(chosen)
    return ds.select(chosen)

train_ds = balanced_sample(imdb["train"], 1000, SEED)
test_ds = balanced_sample(imdb["test"], 500, SEED + 1)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LENGTH = 128

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=[c for c in train_ds.column_names if c != "label"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=[c for c in test_ds.column_names if c != "label"])
for ds in (train_ds, test_ds):
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
start = time.time(); model.train()
for step, batch in enumerate(train_loader, start=1):
    batch = {k:v.to(device) for k,v in batch.items()}
    loss = model(**batch).loss
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if step % 10 == 0 or step == len(train_loader):
        print(f"step {step}/{len(train_loader)} | loss={loss.item():.4f}")
train_seconds = time.time() - start

model.eval(); y_true=[]; y_pred=[]
with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels").cpu().numpy()
        inputs = {k:v.to(device) for k,v in batch.items()}
        preds = model(**inputs).logits.argmax(dim=-1).cpu().numpy()
        y_true.extend(labels.tolist()); y_pred.extend(preds.tolist())
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
accuracy = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
print("\n=== IMDb sentiment classification (test) ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (positive): {precision:.4f}")
print(f"Recall (positive): {recall:.4f}")
print(f"F1 (positive): {f1:.4f}")
print("Confusion matrix [true 0/1 x pred 0/1]:")
print(cm)
print(f"Training time: {train_seconds:.1f}s")


Device: cuda | GPU: Tesla T4


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


step 10/63 | loss=0.7058
step 20/63 | loss=0.6199
step 30/63 | loss=0.5563
step 40/63 | loss=0.6864
step 50/63 | loss=0.4634
step 60/63 | loss=0.3021
step 63/63 | loss=0.5631

=== IMDb sentiment classification (test) ===
Accuracy: 0.8260
Precision (positive): 0.9095
Recall (positive): 0.7240
F1 (positive): 0.8062
Confusion matrix [true 0/1 x pred 0/1]:
[[232  18]
 [ 69 181]]
Training time: 23.8s


In [ ]:
# Bài 20 — trích ví dụ cho error_analysis.md
import torch.nn.functional as F

# Khôi phục đúng thứ tự 500 review test đã dùng ở bài 19.
raw_test = balanced_sample(imdb["test"], 500, SEED + 1)
model.eval()
y_true_20, y_pred_20, conf_20, texts_20 = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].cpu().tolist()
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        probs = torch.softmax(model(**inputs).logits, dim=-1).cpu()
        y_pred_20.extend(probs.argmax(dim=-1).tolist())
        conf_20.extend(probs.max(dim=-1).values.tolist())
        y_true_20.extend(labels)
texts_20 = raw_test["text"]

rows20 = [{"i": i, "text": texts_20[i].replace("\n", " "),
           "true": y_true_20[i], "pred": y_pred_20[i], "confidence": conf_20[i]}
          for i in range(len(y_true_20)) if y_true_20[i] != y_pred_20[i]]
fp20 = sorted([r for r in rows20 if r["true"] == 0 and r["pred"] == 1], key=lambda r: -r["confidence"])
fn20 = sorted([r for r in rows20 if r["true"] == 1 and r["pred"] == 0], key=lambda r: -r["confidence"])
chosen20 = fp20[:3] + fn20[:3]
chosen_ids20 = {r["i"] for r in chosen20}
high20 = [r for r in sorted(rows20, key=lambda r: -r["confidence"]) if r["i"] not in chosen_ids20][:2]

def show_rows20(title, rows):
    print("\n" + title)
    for j, r in enumerate(rows, 1):
        print(f"{j}. index={r['i']} true={'positive' if r['true'] else 'negative'} pred={'positive' if r['pred'] else 'negative'} confidence={r['confidence']:.4f}")
        print(r["text"])

print(f"Device: {device}; wrong={len(rows20)} / {len(y_true_20)}")
show_rows20("3 FALSE POSITIVES (true negative, predicted positive)", fp20[:3])
show_rows20("3 FALSE NEGATIVES (true positive, predicted negative)", fn20[:3])
show_rows20("2 ADDITIONAL HIGHEST-CONFIDENCE WRONG EXAMPLES", high20)


Device: cuda; wrong=87 / 500

3 FALSE POSITIVES (true negative, predicted positive)
1. index=314 true=negative pred=positive confidence=0.8832
This movie is a window on the world of Britain in 1973 - a world of holiday camps, fags and birds. I was actually at Pontins Prestatyn while this was being filmed, so it's fascinating to see where I holidayed as a kid, and what a world we have left behind. The 'plot', such as it is, concerns Stan and Jack's attempts to turn a job at a holiday camp(25 quid a week!) into an opportunity for chasing young women and winding up Blakey. They are joined by the rest of Stan's family and some laughs are to be had from Arthur (Michael Robbins) and Olive's (Anna Karen) persistent chuntering, as well as the sheer oddness of Blakey, but the idea that young women would see anything desirable in brilliantined Stan (Reg Varney) or lecherous Jack (Bob Grant) is just risible. Harmless enough in its time, now it's just a curio from a bygone age.
2. index=124 true=n